# Week Ten | Document Classification
**Sachi Kapoor · DATA 620 Web Analytics**

Can a model learn to separate spam from legitimate email using patterns in previously labeled messages? I use [UCI Spambase](https://doi.org/10.24432/C53G6X), which represents each email with 57 numeric features. These include selected word and character frequencies and statistics about runs of capital letters. The target is 1 for spam and 0 for ham (legitimate mail).

This notebook uses pandas and scikit-learn. It downloads the dataset from UCI when run. The supplied features are already computed from the emails.

In [1]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import pandas as pd
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

url = "https://archive.ics.uci.edu/static/public/94/spambase.zip"
with urlopen(url, timeout=30) as response:
    archive_bytes = response.read()

with ZipFile(BytesIO(archive_bytes)) as archive:
    metadata = archive.read("spambase.names").decode("latin-1")
    feature_names = [
        line.split(":", 1)[0].strip()
        for line in metadata.splitlines()
        if line.strip().endswith("continuous.")
    ]
    with archive.open("spambase.data") as data:
        emails = pd.read_csv(data, header=None, na_values="?")

assert len(feature_names) == 57 and emails.shape[1] == 58
emails.columns = feature_names + ["spam"]
assert set(emails["spam"]) == {0, 1}

print("Original rows:", len(emails))
print("Missing feature values:", emails[feature_names].isna().sum().sum())
print("Identical feature-and-label rows removed:", emails.duplicated().sum())
emails = emails.drop_duplicates()
print(emails["spam"].map({0: "Ham", 1: "Spam"}).value_counts().to_string())

Original rows: 4601
Missing feature values: 0
Identical feature-and-label rows removed: 391
spam
Ham     2531
Spam    1679


## Train and compare three classifiers

I remove identical feature-and-label rows before splitting. I stratify the split into 60% training, 20% development testing, and 20% final testing so that each split has a similar spam proportion. Every classifier uses the same splits. I compare all 57 predictors with the top 10 and top 15. Spambase has 58 columns including the spam label. I rank predictors by their ANOVA F-scores using the training split only.

I compare three approaches from [Chapter 6 of Natural Language Processing with Python](https://www.nltk.org/book/ch06.html): Naive Bayes, decision trees, and maximum entropy. I use scikit-learn for the numeric features, with Gaussian Naive Bayes and binary logistic regression as the maximum entropy model. I use dev-test spam F1 to pick the classifier because it balances spam precision and recall. An always-majority classifier provides a baseline.

In [2]:
from sklearn.base import clone

X_train_dev, X_test, y_train_dev, y_test = train_test_split(
    emails[feature_names], emails["spam"], test_size=0.20,
    stratify=emails["spam"], random_state=42
)
X_train, X_dev, y_train, y_dev = train_test_split(
    X_train_dev, y_train_dev, test_size=0.25,
    stratify=y_train_dev, random_state=42
)
print(f"Train: {len(y_train)} | Dev-test: {len(y_dev)} | Test: {len(y_test)}")

models = {
    "Gaussian Naive Bayes": make_pipeline(SimpleImputer(), GaussianNB()),
    "Decision tree": make_pipeline(
        SimpleImputer(), DecisionTreeClassifier(max_depth=10, random_state=42)
    ),
    "Maximum entropy (logistic regression)": make_pipeline(
        SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000)
    ),
}
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Majority baseline dev-test accuracy: {baseline.score(X_dev, y_dev):.1%}")

ranking_data = SimpleImputer().fit_transform(X_train)
results = []
fitted = {}
for k in (10, 15):
    selector = SelectKBest(f_classif, k=k).fit(ranking_data, y_train)
    chosen = X_train.columns[selector.get_support()].tolist()
    print(f"Top {k} predictors: {', '.join(chosen)}")
for name, model in models.items():
    for k in (10, 15, 57):
        candidate = make_pipeline(
            SimpleImputer(), SelectKBest(f_classif, k=k),
            *(clone(step) for _, step in model.steps[1:])
        )
        candidate.fit(X_train, y_train)
        fitted[name, k] = candidate
        dev_predictions = candidate.predict(X_dev)
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_dev, dev_predictions, average="binary", zero_division=0
        )
        results.append({
            "Classifier": name, "Features": k, "Accuracy": (dev_predictions == y_dev).mean(),
            "Spam precision": precision, "Spam recall": recall, "Spam F1": f1
        })
comparison = pd.DataFrame(results).sort_values("Spam F1", ascending=False, kind="stable")
print(comparison.to_string(index=False, float_format="{:.3f}".format))
winner = comparison.iloc[0]["Classifier"]
winner_k = int(comparison.iloc[0]["Features"])
classifier = fitted[winner, winner_k]
print(f"Selected model: {winner} with {winner_k} features")

Train: 2526 | Dev-test: 842 | Test: 842
Majority baseline dev-test accuracy: 60.1%
Top 10 predictors: word_freq_remove, word_freq_receive, word_freq_free, word_freq_business, word_freq_you, word_freq_your, word_freq_000, word_freq_hp, char_freq_!, char_freq_$
Top 15 predictors: word_freq_our, word_freq_remove, word_freq_receive, word_freq_free, word_freq_business, word_freq_you, word_freq_credit, word_freq_your, word_freq_000, word_freq_hp, word_freq_hpl, word_freq_1999, char_freq_!, char_freq_$, capital_run_length_total
                           Classifier  Features  Accuracy  Spam precision  Spam recall  Spam F1
Maximum entropy (logistic regression)        57     0.942           0.953        0.899    0.925
Maximum entropy (logistic regression)        15     0.910           0.928        0.839    0.881
                        Decision tree        57     0.907           0.913        0.848    0.880
Maximum entropy (logistic regression)        10     0.901           0.923        0.821   

## Evaluate the selected model on the final test set

After selecting the model using development results, I apply it to the reserved test set. The confusion matrix shows missed spam and legitimate messages incorrectly marked as spam.

In [3]:
predicted = classifier.predict(X_test)
baseline_accuracy = baseline.score(X_test, y_test)
accuracy = classifier.score(X_test, y_test)
print(f"Majority baseline test accuracy: {baseline_accuracy:.1%}")
print(f"Selected model test accuracy: {accuracy:.1%}")
print()
print(classification_report(y_test, predicted, labels=[0, 1],
                            target_names=["Ham", "Spam"], digits=3))
matrix = confusion_matrix(y_test, predicted, labels=[0, 1])
print(pd.DataFrame(matrix, index=["Actual ham", "Actual spam"],
                   columns=["Predicted ham", "Predicted spam"]).to_string())

print("\nExample predictions from the held-out set")
examples = pd.DataFrame({"Actual": y_test.to_numpy(), "Predicted": predicted})
print(examples.head(8).replace({0: "Ham", 1: "Spam"}).to_string(index=False))

Majority baseline test accuracy: 60.1%
Selected model test accuracy: 93.5%

              precision    recall  f1-score   support

         Ham      0.931     0.962     0.947       506
        Spam      0.940     0.893     0.916       336

    accuracy                          0.935       842
   macro avg      0.936     0.928     0.931       842
weighted avg      0.935     0.935     0.934       842

             Predicted ham  Predicted spam
Actual ham             487              19
Actual spam             36             300

Example predictions from the held-out set
Actual Predicted
  Spam      Spam
   Ham       Ham
  Spam      Spam
  Spam       Ham
   Ham      Spam
   Ham       Ham
  Spam       Ham
  Spam      Spam


## Findings

The best dev-test spam F1 with 10 predictors was **0.869**. The best dev-test spam F1 with 15 predictors was **0.881**. The best dev-test spam F1 with 57 predictors was **0.925**. I selected **Maximum entropy (logistic regression) with 57 predictors**, which had the highest dev-test spam F1. The printed feature lists show which predictors made the reduced sets. The F-scores rank individual associations with spam, and they do not establish that a predictor causes spam. A smaller set could reduce the work of extracting features from raw email, but I did not measure extraction time on this precomputed dataset.

On the final test set, the selected model correctly classified **787 of 842** emails (**93.5% accuracy**) and had a spam F1 of **0.916**. It caught **300 of 336** spam messages and incorrectly flagged **19** legitimate messages.

This test measures performance on held-out records from the same historical corpus. UCI notes that some legitimate-mail indicators, such as “george” and “650,” reflect the people and location represented in its data. Performance could differ on email from another inbox or time period.

**Reference**  
Hopkins, M., Reeber, E., Forman, G., and Suermondt, J. (1999). [Spambase](https://doi.org/10.24432/C53G6X). UCI Machine Learning Repository.